# 🧩 LLM Red Teaming — Notebook 5: NLI Robustness

**Workstream:** Adversarial robustness of *reasoning* — does the model still infer correctly when the input is crafted to fool it?

Notebooks 01–03 attack the model (evasion, jailbreak, injection) and NB04 measures a harm (bias). This notebook measures a **capability under adversarial pressure**: Natural Language Inference (NLI) — deciding whether a hypothesis is **entailed by**, **neutral to**, or **contradicts** a premise. Unlike NB01, the attack isn't an algorithm we run — the **dataset is the adversary**: ANLI items were written by humans specifically to break strong models.

The headline metric is the **robustness gap** = clean accuracy (MultiNLI) − adversarial accuracy (ANLI / AdvGLUE). A model can score 90%+ on ordinary NLI yet collapse on adversarial NLI; that gap is the reliability story leadership needs.

| Track | Dataset | Role |
|---|---|---|
| Clean baseline | **MultiNLI** (`nyu-mll/multi_nli`) | Ordinary NLI — the reference accuracy |
| Adversarial (human) | **ANLI** R1/R2/R3 (`facebook/anli`) | Human-in-the-loop adversarial NLI; difficulty rises by round |
| Adversarial (perturbed) | **AdvGLUE** (`adv_glue/adv_mnli*`) | Adversarially-perturbed MNLI |

> 🔒 **Security note:** clear all outputs before committing — run outputs can leak the internal target endpoint/model name. Coordinate large runs with the security team.

## Step 0 · Environment Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q \
    openai python-dotenv datasets \
    pandas matplotlib seaborn tqdm pyarrow

print(f'✅ Packages installed into: {sys.executable}')

### 0b — Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv('../.env')

# ── NLI robustness ─────────────────────────────────────────────────────────────
from attacks.robustness import (
    load_nli_dataset, load_mnli, load_anli, load_advglue,
    NLIRunner, NLI_LABELS,
)
from targets import AzureOpenAITarget
from evaluate import (
    nli_summary, nli_accuracy, robustness_gap, confusion_matrix,
    anli_round_curve, error_cases, explain_nli_errors, print_nli_report,
    generate_nli_summary,
)

print('✅ All modules loaded')
print(f'   Label space : {NLI_LABELS}')

---

In [ ]:
# ── ⚙️ CONFIGURATION ────────────────────────────────────────────────────────
# Clean baseline
N_CLEAN       = 100               # MultiNLI validation_matched items

# Adversarial — ANLI (human-crafted; difficulty rises R1 → R3)
ANLI_ROUNDS   = (1, 2, 3)
N_PER_ANLI    = 100               # items per round

# Adversarial — AdvGLUE (perturbed MNLI; small fixed dev sets, taken in full)
ADVGLUE_TASKS = ('mnli', 'mnli_mismatched')

# Sampling & pacing
SHUFFLE   = True
SEED      = 42
SLEEP_SEC = 0.2

# Executive narrative
USE_JUDGE = True                  # judge LLM writes the report narrative (metrics stay deterministic)

RESULTS_DIR = '../results'
CKPT_NLI    = f'{RESULTS_DIR}/05_ckpt_nli.jsonl'   # one resume-safe file, keyed by (source, idx)

n_budget = N_CLEAN + len(ANLI_ROUNDS) * N_PER_ANLI  # + AdvGLUE dev sets (~283) added at load time
print(f'Clean (MNLI)   : {N_CLEAN}')
print(f'ANLI rounds    : {list(ANLI_ROUNDS)} × {N_PER_ANLI}')
print(f'AdvGLUE tasks  : {list(ADVGLUE_TASKS)} (full dev sets)')
print(f'Approx. budget : {n_budget}+ classification calls (one per item)')

### 📚 Background — Why NLI, and why adversarial NLI

**Natural Language Inference (NLI)** is the canonical test of whether a model *reasons* over text rather than pattern-matching surface words. Given a premise, the model must decide if a hypothesis is **entailment**, **neutral**, or **contradiction**. It underpins fact-checking, RAG faithfulness, and contract/policy analysis — anywhere a model must judge whether a claim follows from a source.

**Why adversarial NLI matters.** Frontier models score very high on ordinary NLI, so clean accuracy alone *overstates* reliability. Two benchmarks stress the reasoning directly:

- **ANLI** (Nie et al., 2020) — collected by having humans write hypotheses that *fooled* the best models of the day, across three rounds of escalating difficulty (R1 → R3). Each item ships a human **`reason`** annotation explaining the trap — invaluable for error analysis.
- **AdvGLUE** (Wang et al., 2021) — applies adversarial word/sentence perturbations to GLUE tasks; we use the two MNLI variants so the label space matches.

This is a **reliability / assurance** finding, not a security breach: a large robustness gap means the model's reasoning is brittle under pressure, which maps to **NIST AI 600-1 Information Integrity** and **EU AI Act Art. 15 (accuracy & robustness)** rather than to an attack technique.

### 🔎 Exactly what gets sent — one concrete example of each source (run this)

In [ ]:
# Show the EXACT prompt the model receives, plus one real item from each dataset.
from attacks.robustness.nli import _SYSTEM, _user_prompt

_peek = load_nli_dataset(n_clean=1, anli_rounds=(3,), n_per_anli=1, advglue_tasks=('mnli',))
print('SYSTEM PROMPT sent for every item:')
print('─'*78)
print(_SYSTEM)
print('─'*78)
for src, items in _peek.items():
    it = items[0]
    print(f'\n### {src}  (gold = {NLI_LABELS[it.label]})')
    print(_user_prompt(it.premise[:200], it.hypothesis[:160]))
    if it.reason:
        print(f'   ↳ ANLI reason it is hard: {it.reason[:180]}')

## Step 1 · Instantiate Target & Runner

In [ ]:
target = AzureOpenAITarget()
judge  = AzureOpenAITarget(model=os.getenv('JUDGE_MODEL') or os.getenv('TARGET_MODEL')) if USE_JUDGE else None
runner = NLIRunner(target=target, sleep_sec=SLEEP_SEC)

print('Target configured:', target.__class__.__name__)
print('Judge (report narrative):', 'enabled' if judge else 'disabled (fallback template)')

## Step 2 · Load Datasets (clean + adversarial)

One call builds the whole suite keyed by source. Datasets are fetched once and cached to `eval_datasets/robustness/` (gitignored); a tiny in-repo fallback keeps the notebook runnable offline.

In [ ]:
suite = load_nli_dataset(
    n_clean=N_CLEAN, anli_rounds=ANLI_ROUNDS, n_per_anli=N_PER_ANLI,
    advglue_tasks=ADVGLUE_TASKS, shuffle=SHUFFLE, seed=SEED,
)
rows = [{'source': s, 'kind': 'clean' if s == 'mnli' else 'adversarial',
         'n': len(items), 'has_reason': bool(items and items[0].reason)}
        for s, items in suite.items()]
display(pd.DataFrame(rows))
print(f'Total items to score: {sum(len(v) for v in suite.values())}')

## Step 3 · Sanity Check — Visual Inspection

Classify a handful from each source first and eyeball gold vs. predicted, before spending the full budget.

In [ ]:
sample = {s: items[:3] for s, items in suite.items()}
sanity = runner.run_suite(sample, checkpoint_path=None, verbose=True)
print(f'\nSanity accuracy (tiny sample): {nli_accuracy(sanity):.0%} — expect lower on adversarial sources.')

## Step 4 · Full Evaluation

Runs every item across all sources. **Resume-safe** — re-running picks up from the checkpoint, so an interrupted run costs nothing to continue.

In [ ]:
results = runner.run_suite(suite, checkpoint_path=CKPT_NLI, verbose=False)
print(f'✅ Scored {len(results)} items across {len(suite)} datasets.')

## Step 5 · Robustness Metrics — Accuracy, Gap, ANLI Curve

- **Per-dataset accuracy** — clean vs. adversarial.
- **Robustness gap** — clean − adversarial (the headline; higher = more brittle).
- **ANLI difficulty curve** — accuracy by round (R1 → R3).
- **Confusion matrix** — which way errors go (e.g. neutral misread as contradiction).

In [ ]:
print_nli_report(results)

print('\nPer-dataset summary:'); display(nli_summary(results))
print('Robustness gap:');        display(robustness_gap(results))
print('Confusion matrix (overall, rows = gold, cols = predicted):')
display(confusion_matrix(results))

## Step 6 · Visualisations

In [ ]:
summ = nli_summary(results)
gap  = robustness_gap(results)
curve = anli_round_curve(results)
cm   = confusion_matrix(results)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

# (1) per-dataset accuracy — clean vs adversarial
colors = ['#2E7D32' if k == 'clean' else '#C62828' for k in summ['kind']]
ax[0].barh(summ['source'], summ['accuracy'], color=colors)
ax[0].set_xlim(0, 1); ax[0].invert_yaxis()
ax[0].set_title('Accuracy by dataset (green = clean)'); ax[0].set_xlabel('accuracy')
for y, v in enumerate(summ['accuracy']):
    ax[0].text(v + 0.01, y, f'{v:.0%}', va='center', fontsize=9)

# (2) ANLI difficulty curve
if not curve.empty:
    ax[1].plot(curve['round'], curve['accuracy'], 'o-', color='#C62828', lw=2, ms=8)
    ax[1].set_ylim(0, 1); ax[1].set_title('ANLI difficulty curve (R1→R3)')
    ax[1].set_ylabel('accuracy'); ax[1].grid(alpha=.3)
    for x, v in zip(curve['round'], curve['accuracy']):
        ax[1].text(x, v + 0.03, f'{v:.0%}', ha='center', fontsize=9)

# (3) confusion heatmap (drop all-zero unparsed col if empty)
cmp = cm.drop(columns=['unparsed']) if cm['unparsed'].sum() == 0 else cm
sns.heatmap(cmp, annot=True, fmt='d', cmap='Blues', ax=ax[2], cbar=False)
ax[2].set_title('Confusion (gold → predicted)'); ax[2].set_ylabel('gold'); ax[2].set_xlabel('predicted')

plt.tight_layout(); plt.show()

## Step 7 · Error-Case Analysis

Where the model failed, and — for ANLI — *why the item is hard*, straight from the human annotator who wrote it. These are the most instructive cases for understanding the reasoning weakness.

In [ ]:
# ANLI errors first (they carry the human 'reason'), then AdvGLUE.
explain_nli_errors(results, source='anli_r3', n=4)
explain_nli_errors(results, source='anli_r2', n=2)
print('\n— AdvGLUE perturbation errors —')
explain_nli_errors(results, source='advglue_mnli', n=2)

## Step 8 · Industry & Regulatory Alignment

Reasoning robustness is an **assurance** property, so it maps to integrity/accuracy obligations rather than attack catalogues.

| Framework | Reference | Why it applies |
|---|---|---|
| **NIST AI 600-1** | §2.5 Information Integrity · §2.2 Confabulation | A model that mis-infers entailment under adversarial input can assert false conclusions as true |
| **MITRE ATLAS** | AML.T0043 (Craft Adversarial Data) | ANLI/AdvGLUE are adversarial-example evasion of an NLP capability |
| **OWASP LLM Top 10** | LLM09 Misinformation | Brittle reasoning surfaces as confidently wrong outputs |
| **EU AI Act** | Art. 15 (accuracy, robustness & cybersecurity) | High-risk systems must perform consistently and resist adversarial manipulation |

> Robustness gaps are a *reliability* signal: a large gap means clean-set accuracy overstates how the model behaves on hard, adversarial, or out-of-distribution reasoning — exactly the inputs a production system meets in the wild.

## Step 9 · Executive Report

Deterministic metrics + a judge-LLM narrative, rendered as a business-level HTML report with the illustrative-sample disclaimer. The judge writes only the *narrative* — every number is computed deterministically.

In [ ]:
from IPython.display import HTML
exec_html, exec_data = generate_nli_summary(
    results,
    target=judge or target,
    config={'model_name': 'GPT-5-4 (Azure)', 'run_date': str(pd.Timestamp.today().date())},
)
HTML(exec_html)

## Step 10 · Save Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame([r.__dict__ for r in results]).to_csv(f'{RESULTS_DIR}/05_nli_results.csv', index=False)
nli_summary(results).to_csv(f'{RESULTS_DIR}/05_nli_summary.csv', index=False)
robustness_gap(results).to_csv(f'{RESULTS_DIR}/05_robustness_gap.csv', index=False)
# Error audit (premise/hypothesis/gold/pred + ANLI reason) for human review
errs = error_cases(results, source=None, n=10_000)
if errs:
    pd.DataFrame(errs).to_csv(f'{RESULTS_DIR}/05_error_cases.csv', index=False)
with open(f'{RESULTS_DIR}/05_executive_summary.html', 'w') as f:
    f.write(exec_html)
print(f'Saved results, summary, robustness gap, {len(errs)} error case(s), + executive report → {RESULTS_DIR}/')

## Step 11 · Observations & Key Takeaways

Computed from *this* run — not hardcoded.

In [ ]:
summ = nli_summary(results)
gap  = robustness_gap(results)
curve = anli_round_curve(results)
clean = summ[summ['kind'] == 'clean']['accuracy']
clean_acc = float(clean.iloc[0]) if len(clean) else float('nan')
adv = summ[summ['kind'] == 'adversarial']
worst = gap.loc[gap['robustness_gap'].idxmax()] if not gap.empty else None

print('── NLI ROBUSTNESS — KEY TAKEAWAYS ──')
print(f'  Clean accuracy (MNLI)        : {clean_acc:.1%}')
print(f'  Adversarial accuracy (mean)  : {adv["accuracy"].mean():.1%}  over {len(adv)} adversarial set(s)')
if worst is not None:
    print(f'  Worst robustness gap         : {worst["source"]}  {worst["robustness_gap"]:+.1%} '
          f'(clean {worst["clean_acc"]:.0%} → adv {worst["adv_acc"]:.0%})')
if not curve.empty:
    trend = 'falls' if curve['accuracy'].iloc[0] > curve['accuracy'].iloc[-1] else 'holds/rises'
    print(f'  ANLI curve R1→R3             : {" → ".join(f"{v:.0%}" for v in curve["accuracy"])}  ({trend} with difficulty)')
unparsed = int((pd.DataFrame([r.__dict__ for r in results])['pred'] == -1).sum())
print(f'  Unparseable replies          : {unparsed} / {len(results)}  (counted as wrong)')

big_gap = gap[gap['robustness_gap'] >= 0.25] if not gap.empty else gap
print()
if not gap.empty and not big_gap.empty:
    print(f'⚠️  {len(big_gap)} dataset(s) show a robustness gap ≥ 25% — clean accuracy materially')
    print('   overstates reliability on adversarial reasoning. Add adversarial NLI to the eval suite (NIST')
    print('   Information Integrity · EU AI Act Art. 15). Inspect Step 7 error cases for the failure modes.')
else:
    print('✅ No dataset shows a robustness gap ≥ 25%. Reasoning degrades gracefully under adversarial')
    print('   pressure; continue monitoring the ANLI curve as harder rounds / new benchmarks are added.')